In [3]:
import torch
import random
import evaluate
import os
from torch.utils.data import DataLoader
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    DataCollatorForTokenClassification, 
    AutoConfig, 
    set_seed
)
from tqdm.auto import tqdm

# Set seed and environment
set_seed(42)
os.environ["HF_TOKEN"] = "your_token_here" # It's better to use an env variable than hardcoding!

# 1. Load the BC5CDR dataset directly from Hugging Face
print("Loading BC5CDR dataset...")
raw_datasets = load_dataset("tner/bc5cdr", trust_remote_code=True)

# Define hyperparameters
learning_rate = 2e-5
num_train_epochs = 3
model_name = "google-bert/bert-base-cased"
batch_size = 8

# 2. Extract label information manually
print("Extracting unique tags from dataset...")

# Collect all unique tag IDs from the training set
unique_tag_ids = set()
for example in raw_datasets["train"]:
    unique_tag_ids.update(example["tags"])

# Convert IDs to a sorted list
# Note: In tner/bc5cdr, these are usually already integers (0, 1, 2, 3, 4)
sorted_tag_ids = sorted(list(unique_tag_ids))

# BC5CDR mapping for this dataset version:
# 0: O, 1: B-Chemical, 2: I-Chemical, 3: B-Disease, 4: I-Disease
label_list = ["O", "B-Chemical", "I-Chemical", "B-Disease", "I-Disease"]

# If the dataset uses more or different tags, you can use this fallback:
if len(sorted_tag_ids) != len(label_list):
    label_list = [str(i) for i in sorted_tag_ids]

tag2id = {tag: i for i, tag in enumerate(label_list)}
id2tag = {i: tag for i, tag in enumerate(label_list)}

text_column_name = "tokens"
label_column_name = "tags"



# Read tokenizer and model configuration
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2tag, 
    label2id=tag2id
)

# 3. Tokenization and Alignment logic (remains mostly the same)
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples[text_column_name],
        max_length=128,             
        padding=False,              
        truncation=True, 
        is_split_into_words=True
    )

    all_labels = []
    for batch_index, labels in enumerate(examples[label_column_name]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        label_ids = []
        prev_word_id = None
        
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) 
            elif word_id == prev_word_id:
                label_ids.append(-100)
            else:
                label_ids.append(labels[word_id])
            prev_word_id = word_id
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Process the dataset
processed_raw_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Running tokenizer"
)

# Create Dataloaders
train_dataset = processed_raw_datasets["train"]
eval_dataset = processed_raw_datasets["validation"]
test_dataset = processed_raw_datasets["test"]

model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=data_collator, batch_size=batch_size)
eval_dataloader = DataLoader(eval_dataset, collate_fn=data_collator, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, collate_fn=data_collator, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- Training Loop ---
print("Starts training...")
for epoch in range(num_train_epochs):
    model.train()
    pbar = tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})

# --- Evaluation ---
metric = evaluate.load("seqeval")

def get_labels(predictions, references):
    true_predictions = []
    true_labels = []
    for pred_seq, ref_seq in zip(predictions, references):
        current_preds = [id2tag[p.item()] for p, r in zip(pred_seq, ref_seq) if r != -100]
        current_refs = [id2tag[r.item()] for p, r in zip(pred_seq, ref_seq) if r != -100]
        true_predictions.append(current_preds)
        true_labels.append(current_refs)
    return true_predictions, true_labels

print("Evaluating...")
model.eval()
all_predictions = []
all_labels = []

for batch in tqdm(eval_dataloader, desc="Evaluating"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    
    predictions = outputs.logits.argmax(dim=-1)
    predicted_labels, true_labels = get_labels(predictions, batch["labels"])
    all_predictions.extend(predicted_labels)
    all_labels.extend(true_labels)

results = metric.compute(predictions=all_predictions, references=all_labels)
print(f"Validation Results: {results['overall_f1']}")

# --- Generate Test File (IOB2 format) ---
print("Writing test_predictions.iob2...")
with open("test_predictions.iob2", "w", encoding="utf-8") as f:
    # We iterate over the original test set to get raw tokens
    for i, example in enumerate(tqdm(raw_datasets["test"], desc="Predicting Test")):
        tokens = example["tokens"]
        inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
        word_ids = inputs.word_ids(batch_index=0)
        
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        
        predictions = outputs.logits.argmax(dim=-1)[0]
        
        final_labels = []
        prev_word_id = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None and word_id != prev_word_id:
                final_labels.append(id2tag[predictions[idx].item()])
                prev_word_id = word_id
        
        for idx, (token, label) in enumerate(zip(tokens, final_labels), 1):
            f.write(f"{idx}\t{token}\t{label}\n")
        f.write("\n")

Loading BC5CDR dataset...
Extracting unique tags from dataset...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6338.48it/s]
BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Starts training...


Training Epoch 3: 100%|██████████| 654/654 [34:06<00:00,  3.13s/it, loss=0.00314] 


Evaluating...


Evaluating: 100%|██████████| 667/667 [09:32<00:00,  1.17it/s]


Validation Results: 0.8599783904918163
Writing test_predictions.iob2...


Predicting Test: 100%|██████████| 5865/5865 [10:46<00:00,  9.07it/s] 
